# 🏆 Top Artists & Tracks

Rankings der meistgehörten Künstler und Songs – gesamt und pro Jahr.

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from config import *
from data_loader import load_data, get_music, ensure_results_dir

plt.style.use(PLOT_STYLE)
plt.rcParams['figure.dpi'] = PLOT_DPI
plt.rcParams['savefig.dpi'] = PLOT_DPI
plt.rcParams['savefig.bbox'] = 'tight'

In [ ]:
df = load_data()
music = get_music(df)

## Top Artists – Gesamte Hörzeit (alle Jahre)

In [ ]:
top_artists = (music.groupby('artist')['hours_played'].sum()
               .sort_values(ascending=False).head(TOP_N_ARTISTS).reset_index())

fig, ax = plt.subplots(figsize=PLOT_FIGSIZE_TALL)
bars = ax.barh(range(len(top_artists)), top_artists['hours_played'], color=COLOR_PRIMARY)
ax.set_yticks(range(len(top_artists)))
ax.set_yticklabels(top_artists['artist'])
ax.invert_yaxis()
for bar, h in zip(bars, top_artists['hours_played']):
    ax.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height()/2,
            f'{h:,.1f}h', va='center', fontsize=9)
ax.set_xlabel('Stunden')
ax.set_title(f'🎤 Top {TOP_N_ARTISTS} Artists (Gesamte Hörzeit)', fontsize=16, fontweight='bold')
plt.tight_layout()

out = ensure_results_dir('overview')
fig.savefig(out / 'top_artists_alltime.png')
plt.show()

## Top Tracks – Gesamte Hörzeit (alle Jahre)

In [ ]:
top_tracks = (music.groupby(['track', 'artist'])['hours_played'].sum()
              .sort_values(ascending=False).head(TOP_N_TRACKS).reset_index())
top_tracks['label'] = top_tracks['track'] + ' – ' + top_tracks['artist']

fig, ax = plt.subplots(figsize=PLOT_FIGSIZE_TALL)
bars = ax.barh(range(len(top_tracks)), top_tracks['hours_played'], color=COLOR_ACCENT)
ax.set_yticks(range(len(top_tracks)))
ax.set_yticklabels(top_tracks['label'], fontsize=9)
ax.invert_yaxis()
for bar, h in zip(bars, top_tracks['hours_played']):
    ax.text(bar.get_width() + 0.1, bar.get_y() + bar.get_height()/2,
            f'{h:,.1f}h', va='center', fontsize=9)
ax.set_xlabel('Stunden')
ax.set_title(f'🎶 Top {TOP_N_TRACKS} Tracks (Gesamte Hörzeit)', fontsize=16, fontweight='bold')
plt.tight_layout()
fig.savefig(out / 'top_tracks_alltime.png')
plt.show()

## Top Artists & Tracks – Pro Jahr

In [ ]:
years = sorted(music['year'].unique())

for year in years:
    year_music = music[music['year'] == year]
    year_out = ensure_results_dir(year)
    top_n = min(TOP_N_ARTISTS, 15)  # etwas weniger für Jahres-Plots

    # Top Artists
    ya = (year_music.groupby('artist')['hours_played'].sum()
          .sort_values(ascending=False).head(top_n).reset_index())

    fig, axes = plt.subplots(1, 2, figsize=(18, max(8, top_n * 0.5)))

    axes[0].barh(range(len(ya)), ya['hours_played'], color=COLOR_PRIMARY)
    axes[0].set_yticks(range(len(ya)))
    axes[0].set_yticklabels(ya['artist'])
    axes[0].invert_yaxis()
    for i, h in enumerate(ya['hours_played']):
        axes[0].text(h + 0.1, i, f'{h:.1f}h', va='center', fontsize=9)
    axes[0].set_xlabel('Stunden')
    axes[0].set_title(f'🎤 Top {top_n} Artists', fontweight='bold')

    # Top Tracks
    yt = (year_music.groupby(['track', 'artist'])['hours_played'].sum()
          .sort_values(ascending=False).head(top_n).reset_index())
    yt['label'] = yt['track'].str[:30] + ' – ' + yt['artist'].str[:20]

    axes[1].barh(range(len(yt)), yt['hours_played'], color=COLOR_ACCENT)
    axes[1].set_yticks(range(len(yt)))
    axes[1].set_yticklabels(yt['label'], fontsize=8)
    axes[1].invert_yaxis()
    for i, h in enumerate(yt['hours_played']):
        axes[1].text(h + 0.05, i, f'{h:.1f}h', va='center', fontsize=9)
    axes[1].set_xlabel('Stunden')
    axes[1].set_title(f'🎶 Top {top_n} Tracks', fontweight='bold')

    plt.suptitle(f'🏆 Top Artists & Tracks {year}', fontsize=16, fontweight='bold', y=1.02)
    plt.tight_layout()
    fig.savefig(year_out / 'top_artists_tracks.png')
    plt.show()
    print(f"Gespeichert: {year_out / 'top_artists_tracks.png'}")

## Artist-Treue: Wer taucht jedes Jahr auf?

In [ ]:
artist_years = music.groupby('artist')['year'].nunique().reset_index()
artist_years.columns = ['artist', 'years_active']
total_years = music['year'].nunique()

# Artists die in JEDEM Jahr vorkommen
loyal = artist_years[artist_years['years_active'] == total_years]
loyal_hours = (music[music['artist'].isin(loyal['artist'])]
               .groupby('artist')['hours_played'].sum()
               .sort_values(ascending=False).head(20).reset_index())

print(f"\n{len(loyal)} Artists in allen {total_years} Jahren gehört:")
print(f"Top 20 davon nach Hörzeit:\n")

fig, ax = plt.subplots(figsize=PLOT_FIGSIZE_TALL)
ax.barh(range(len(loyal_hours)), loyal_hours['hours_played'], color=COLOR_PALETTE[4])
ax.set_yticks(range(len(loyal_hours)))
ax.set_yticklabels(loyal_hours['artist'])
ax.invert_yaxis()
for i, h in enumerate(loyal_hours['hours_played']):
    ax.text(h + 0.2, i, f'{h:.1f}h', va='center', fontsize=9)
ax.set_xlabel('Stunden (gesamt)')
ax.set_title(f'💎 Treueste Artists (in allen {total_years} Jahren gehört)', fontsize=14, fontweight='bold')
plt.tight_layout()
fig.savefig(out / 'loyal_artists.png')
plt.show()